# Gráficos dos resultados já executados

Este notebook é **somente leitura**. Ele não baixa dados, não executa benchmarks, não limpa diretórios, não altera o SQLite e não exporta arquivos. Sua única fonte é `results/raw/all_benchmarks.csv`, produzido previamente pelo notebook experimental.

Os gráficos mostram as cinco operações solicitadas — busca por ID, categoria e nome, inserção e remoção — para ordens 32, 128 e 256, nos volumes de 10 mil, 100 mil e 700 mil filmes. Runs individuais nunca são desenhadas: todos os valores apresentados são médias em milissegundos.


## Localização e proteção da fonte

Primeiro localizamos a raiz do projeto e registramos tamanho e horário de modificação do CSV. Esses valores serão conferidos novamente no final para demonstrar que a execução não alterou o arquivo.


In [ ]:
from pathlib import Path
import sys

candidate = Path.cwd().resolve()
PROJECT_ROOT = candidate if (candidate / "src").exists() else candidate.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

RESULTS_FILE = PROJECT_ROOT / "results" / "raw" / "all_benchmarks.csv"
if not RESULTS_FILE.exists():
    raise FileNotFoundError(
        "Resultados não encontrados. Execute primeiro "
        "notebooks/experimento_arvores_b_bplus.ipynb."
    )
source_before = (RESULTS_FILE.stat().st_size, RESULTS_FILE.stat().st_mtime_ns)
print(f"Fonte: {RESULTS_FILE.relative_to(PROJECT_ROOT)}")
print(f"Tamanho: {source_before[0]:,} bytes")


## Leitura e validação

O CSV é carregado em memória e validado antes de qualquer gráfico. A execução é interrompida se faltar uma operação, ordem ou tamanho, evitando apresentar resultados parciais como se estivessem completos.


In [ ]:
import pandas as pd
from IPython.display import Markdown, display

from src.benchmark import summarize_results
from src.plots import create_figures, create_results_overview_figures

ORDENS_ESPERADAS = {32, 128, 256}
TAMANHOS_ESPERADOS = {10_000, 100_000, 700_000}
OPERACOES_ESPERADAS = {
    "search_by_id", "search_by_category", "search_by_title",
    "insert_catalog", "delete_catalog",
}

raw_results = pd.read_csv(RESULTS_FILE, low_memory=False)
operacoes_encontradas = set(raw_results["operation"].dropna())
if operacoes_encontradas != OPERACOES_ESPERADAS or "time_ms" not in raw_results:
    raise RuntimeError(
        "O CSV pertence ao formato anterior ou está incompleto. Reinicie o kernel "
        "e execute notebooks/experimento_arvores_b_bplus.ipynb para gerar as cinco "
        "operações atuais em milissegundos."
    )
assert set(raw_results["order"].dropna().astype(int)) == ORDENS_ESPERADAS
assert set(raw_results["sample_size"].dropna().astype(int)) == TAMANHOS_ESPERADOS
assert "time_ns" not in raw_results.columns
assert raw_results.groupby(
    ["operation", "structure", "order", "sample_size"]
).size().eq(300).all(), "Cada combinação deve conter exatamente 300 medições."
print(f"Medições carregadas: {len(raw_results):,}")
display(raw_results.groupby(["operation", "structure"]).size().rename("medições").to_frame())


## Preparação das médias

Para não misturar cenários, os gráficos usam apenas a carga com inserção aleatória. Caso o CSV também contenha inserção ordenada, ela não é exibida. Isso evita linhas duplicadas e mantém a comparação direta.


In [ ]:
summary = summarize_results(raw_results)
figures = create_figures(raw_results, summary)
figures.update(create_results_overview_figures(raw_results))
assert set(figures) == {
    "01_search_by_id", "02_search_by_category", "03_search_by_title",
    "04_insert_catalog", "05_delete_catalog",
    "06_best_largest_size", "07_structure_comparison", "08_scalability",
}


## 1. Resumo direto

Este primeiro gráfico responde à pergunta mais prática: **com 700 mil filmes, qual é o menor tempo obtido por cada estrutura?** Há somente duas barras por operação. O rótulo informa a média em milissegundos e a ordem que produziu esse resultado; cada painel possui sua própria escala vertical para preservar a legibilidade.


In [ ]:
figures["06_best_largest_size"].show()


O gráfico seguinte resume quem venceu em quinze células. Azul indica vantagem da Árvore B+; cinza indica vantagem da Árvore B. O percentual mede quanto a vencedora foi mais rápida que a outra estrutura, depois de escolher a melhor ordem de cada uma. Passe o mouse para ver tempos em milissegundos e ordens.


In [ ]:
figures["07_structure_comparison"].show()


## 2. Efeito da ordem e do tamanho

Os cinco mapas abaixo preservam o detalhe das 18 combinações de cada operação. Árvore B fica à esquerda e Árvore B+ à direita; colunas são ordens e linhas são tamanhos. Cada célula mostra **uma única média em milissegundos**, nunca uma run individual. Tons mais escuros indicam maior tempo e ★ marca a melhor combinação dentro de cada estrutura. **Menor é melhor.**


### Busca por ID

Mede a localização exata de um filme por sua chave identificadora. Observe como o aumento do catálogo e da ordem altera o caminho de busca em cada estrutura.


In [ ]:
figures["01_search_by_id"].show()


### Busca por categoria

Mede a recuperação dos filmes associados a uma categoria. Essa consulta pode retornar vários registros, por isso tende a custar mais que uma busca pontual.


In [ ]:
figures["02_search_by_category"].show()


### Busca pelo nome do filme

Mede a localização de títulos normalizados, preservando filmes diferentes que possuem o mesmo nome.


In [ ]:
figures["03_search_by_title"].show()


### Inserção no catálogo

Mede a inclusão de um novo identificador no índice primário. Cada repetição começa com a árvore original para que todas as 150 inserções sejam realmente novas.


In [ ]:
figures["04_insert_catalog"].show()


### Remoção do catálogo

Mede a exclusão de identificadores existentes, incluindo empréstimos, fusões e eventual redução da raiz necessários para manter a árvore válida.


In [ ]:
figures["05_delete_catalog"].show()


## 3. Crescimento com o catálogo

Para mostrar a tendência sem excesso de linhas, cada painel contém apenas Árvore B e Árvore B+. A ordem fica fixa em cada linha e é escolhida pela menor média geral daquela operação. As escalas verticais são independentes, pois busca por categoria e busca pontual possuem custos muito diferentes.


In [ ]:
figures["08_scalability"].show()


## Verificação de integridade

A última célula confirma que o CSV permaneceu com o mesmo tamanho e horário de modificação durante toda a execução. Nenhuma função de escrita é chamada neste notebook.


In [ ]:
source_after = (RESULTS_FILE.stat().st_size, RESULTS_FILE.stat().st_mtime_ns)
assert source_after == source_before, "O arquivo de resultados foi alterado durante a leitura."
print("Integridade confirmada: nenhum dado foi alterado.")
